In [60]:
import pandas as pd
import numpy as np

In [61]:
admissions = pd.read_csv(r"D:\OneDrive\Documents\Downloads\Hospital_Project\admissions_raw.csv")

In [62]:
admissions.head()

,admission_id,patient_id,hospital_id,department_id,primary_doctor_id,admission_date,discharge_date,admission_type,diagnosis,severity_level,discharge_status,readmitted_30_days
0,1,3415,8,4,123,2024-03-10,2024-03-16,Emergency,Anemia,Critical,Home,Yes
1,2,3610,8,1,209,2023-05-19,2023-05-20,Planned,Arrhythmia,Moderate,Home,No
2,3,7584,8,3,190,2025-06-16,2025-06-18,Emergency,Joint Replacement,Moderate,Home,Yes
3,4,4459,2,3,176,2025-10-15,2025-10-16,Referral,Joint Replacement,Moderate,Home,No
4,5,5508,3,12,140,2024-05-07,2024-05-23,Planned,Multi Organ Dysfunction,Moderate,Home,No


In [63]:
admissions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30130 entries, 0 to 30129
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   admission_id        30130 non-null  int64 
 1   patient_id          30130 non-null  int64 
 2   hospital_id         30130 non-null  int64 
 3   department_id       30130 non-null  int64 
 4   primary_doctor_id   30130 non-null  int64 
 5   admission_date      30130 non-null  object
 6   discharge_date      29879 non-null  object
 7   admission_type      30130 non-null  object
 8   diagnosis           29987 non-null  object
 9   severity_level      30130 non-null  object
 10  discharge_status    30130 non-null  object
 11  readmitted_30_days  30130 non-null  object
dtypes: int64(5), object(7)
memory usage: 2.8+ MB


In [64]:
admissions.isnull().sum()

admission_id            0
patient_id              0
hospital_id             0
department_id           0
primary_doctor_id       0
admission_date          0
discharge_date        251
admission_type          0
diagnosis             143
severity_level          0
discharge_status        0
readmitted_30_days      0
dtype: int64

In [65]:
admissions = admissions.drop_duplicates()

In [66]:
len(admissions)

30000

In [67]:
admissions['admission_date'] = pd.to_datetime(
    admissions['admission_date'],
    format='mixed',
    dayfirst=True
)

admissions['discharge_date'] = pd.to_datetime(
    admissions['discharge_date'],
    format='mixed',
    dayfirst=True
)

In [68]:
admissions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30000 entries, 0 to 29999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   admission_id        30000 non-null  int64         
 1   patient_id          30000 non-null  int64         
 2   hospital_id         30000 non-null  int64         
 3   department_id       30000 non-null  int64         
 4   primary_doctor_id   30000 non-null  int64         
 5   admission_date      30000 non-null  datetime64[ns]
 6   discharge_date      29750 non-null  datetime64[ns]
 7   admission_type      30000 non-null  object        
 8   diagnosis           29860 non-null  object        
 9   severity_level      30000 non-null  object        
 10  discharge_status    30000 non-null  object        
 11  readmitted_30_days  30000 non-null  object        
dtypes: datetime64[ns](2), int64(5), object(5)
memory usage: 3.0+ MB


In [70]:
admissions['length_of_stay'] = (
    admissions['discharge_date']
    - admissions['admission_date']
).dt.days

In [82]:
admissions[['discharge_date', 'length_of_stay']].isnull().sum()

discharge_date    250
length_of_stay      0
dtype: int64

In [83]:
admissions['admission_date'].isnull().sum()

np.int64(0)

In [72]:
median_stay = admissions['length_of_stay'].median()

In [73]:
admissions['length_of_stay'] = admissions[
    'length_of_stay'
].fillna(median_stay)

In [84]:
admissions['discharge_date'] = admissions[
    'discharge_date'
].fillna(
    admissions['admission_date']
    + pd.to_timedelta(
        admissions['length_of_stay'],
        unit='D'
    )
)

In [74]:
admissions['diagnosis'] = admissions[
    'diagnosis'
].fillna('Unknown')

In [75]:
admissions['admission_type'] = admissions[
    'admission_type'
].fillna('Unknown')

In [76]:
admissions['severity_level'] = admissions[
    'severity_level'
].fillna('Unknown')

In [77]:
admissions['discharge_status'] = admissions[
    'discharge_status'
].fillna('Unknown')

In [78]:
admissions['readmitted_30_days'] = admissions[
    'readmitted_30_days'
].fillna('Unknown')

In [79]:
text_columns = [
    'admission_type',
    'diagnosis',
    'severity_level',
    'discharge_status',
    'readmitted_30_days'
]

for col in text_columns:
    admissions[col] = admissions[col].str.strip()

In [80]:
admissions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30000 entries, 0 to 29999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   admission_id        30000 non-null  int64         
 1   patient_id          30000 non-null  int64         
 2   hospital_id         30000 non-null  int64         
 3   department_id       30000 non-null  int64         
 4   primary_doctor_id   30000 non-null  int64         
 5   admission_date      30000 non-null  datetime64[ns]
 6   discharge_date      29750 non-null  datetime64[ns]
 7   admission_type      30000 non-null  object        
 8   diagnosis           30000 non-null  object        
 9   severity_level      30000 non-null  object        
 10  discharge_status    30000 non-null  object        
 11  readmitted_30_days  30000 non-null  object        
 12  length_of_stay      30000 non-null  float64       
dtypes: datetime64[ns](2), float64(1), int64(5), object(

In [85]:
admissions.to_csv("cleaned_admissions.csv",index=False)